In [2]:
"""
Análisis de complejidad en señales EEG de sueño
=================================================
Dataset: Sleep-EDF Expanded (PhysioNet) - Kemp et al. 2000
https://physionet.org/content/sleep-edfx/

El dataset contiene registros PSG de noche completa (EEG, EOG, EMG) junto
con hipnogramas (anotaciones de etapas de sueño: W, N1, N2, N3, REM).
Es uno de los datasets más usados en la literatura para estudios de
complejidad de señales EEG durante el sueño (entropía, dimensión fractal, etc.)

Requisitos:
    pip install mne antropy numpy pandas matplotlib

MNE incluye un descargador oficial que trae los archivos EDF directamente
desde PhysioNet, por lo que no hace falta manejar URLs a mano.
"""
!pip install -q mne antropy

import numpy as np
import pandas as pd
import mne
from mne.datasets.sleep_physionet.age import fetch_data
import antropy as ant  # librería especializada en métricas de complejidad/entropía

# --------------------------------------------------------------------------
# 1. Descarga del dataset (un sujeto, una noche, para mantenerlo liviano)
# --------------------------------------------------------------------------
SUBJECT = [0]      # índice del sujeto (0 a 82 disponibles)
RECORDING = [1]    # noche 1

paths = fetch_data(subjects=SUBJECT, recording=RECORDING)
psg_path, hypnogram_path = paths[0]

print(f"PSG:       {psg_path}")
print(f"Hipnograma:{hypnogram_path}")

# --------------------------------------------------------------------------
# 2. Carga de la señal EEG y las anotaciones de etapas de sueño
# --------------------------------------------------------------------------
raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
annotations = mne.read_annotations(hypnogram_path)
raw.set_annotations(annotations, emit_warning=False)

# Nos quedamos con un canal EEG estándar del dataset (Fpz-Cz)
raw.pick(["EEG Fpz-Cz"])

# Mapeo de las anotaciones originales a un set reducido de etapas AASM
mapping = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N3",
    "Sleep stage R": "REM",
}
event_id = {v: i for i, v in enumerate(sorted(set(mapping.values())))}
events, _ = mne.events_from_annotations(
    raw, event_id={k: event_id[v] for k, v in mapping.items() if k in [a for a in annotations.description]},
    chunk_duration=30.0,  # cada época de sueño dura 30s por convención
)

epochs = mne.Epochs(
    raw, events, event_id=event_id, tmin=0, tmax=30.0 - 1 / raw.info["sfreq"],
    baseline=None, preload=True, verbose=False,
)

# --------------------------------------------------------------------------
# 3. Cálculo de métricas de complejidad por época y por etapa de sueño
# --------------------------------------------------------------------------
inv_event_id = {v: k for k, v in event_id.items()}
data = epochs.get_data()[:, 0, :]  # (n_epocas, n_muestras) canal Fpz-Cz
labels = epochs.events[:, -1]

resultados = []
for signal, label in zip(data, labels):
    resultados.append({
        "etapa": inv_event_id[label],
        "perm_entropy": ant.perm_entropy(signal, normalize=True),
        "sample_entropy": ant.sample_entropy(signal),
        "higuchi_fd": ant.higuchi_fd(signal),
        "spectral_entropy": ant.spectral_entropy(
            signal, sf=raw.info["sfreq"], method="welch", normalize=True
        ),
    })

df = pd.DataFrame(resultados)

# --------------------------------------------------------------------------
# 4. Resumen: complejidad promedio por etapa de sueño
# --------------------------------------------------------------------------
resumen = df.groupby("etapa").mean(numeric_only=True)
print("\nComplejidad promedio por etapa de sueño:")
print(resumen)

df.to_csv("eeg_sleep_complexity_por_epoca.csv", index=False)
resumen.to_csv("eeg_sleep_complexity_resumen.csv")
print("\nArchivos guardados: eeg_sleep_complexity_por_epoca.csv, eeg_sleep_complexity_resumen.csv")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 40.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Using default location ~/mne_data for PHYSIONET_SLEEP...
Creating /root/mne_data


  0%|                                              | 0.00/48.3M [00:00<?, ?B/s]

  0%|                                              | 0.00/4.62k [00:00<?, ?B/s]

Download complete in 03m06s (46.1 MB)
PSG:       /root/mne_data/physionet-sleep-data/SC4001E0-PSG.edf
Hipnograma:/root/mne_data/physionet-sleep-data/SC4001EC-Hypnogram.edf


/tmp/ipykernel_6390/855310751.py:41: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
/tmp/ipykernel_6390/855310751.py:41: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
/tmp/ipykernel_6390/855310751.py:41: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)


Used Annotations descriptions: [np.str_('Sleep stage 1'), np.str_('Sleep stage 2'), np.str_('Sleep stage 3'), np.str_('Sleep stage 4'), np.str_('Sleep stage R'), np.str_('Sleep stage W')]

Complejidad promedio por etapa de sueño:
       perm_entropy  sample_entropy  higuchi_fd  spectral_entropy
etapa                                                            
N1         0.928542        1.075132    1.515017          0.634246
N2         0.873815        0.723310    1.365843          0.553069
N3         0.802642        0.498753    1.228112          0.458118
REM        0.927302        1.042598    1.473984          0.620675
W          0.988170        0.925981    1.594773          0.536449

Archivos guardados: eeg_sleep_complexity_por_epoca.csv, eeg_sleep_complexity_resumen.csv


Actúa como un programador python y experto en neurociencias. Escribe un código que cargue un dataset desde la web sobre analisis de complejidad en señales de EEG en sueño.
Algunos lugares para buscar datos:

https://datos.gob.ar/
https://huggingface.co/
https://www.kaggle.com/datasets

Respuesta:
Usé el Sleep-EDF Expanded (PhysioNet/OpenNeuro), el dataset de referencia en la literatura de complejidad de EEG de sueño: 197 registros PSG de noche completa con hipnogramas de etapas W/N1/N2/N3/REM. Lo elegí en vez de Kaggle/HF porque mne trae un descargador oficial (fetch_data) que lo baja directo desde PhysioNet sin manejar URLs a mano, y antropy ya implementa las métricas de complejidad estándar.

Qué hace el script:

Descarga PSG + hipnograma de un sujeto/noche con mne.datasets.sleep_physionet.age.fetch_data.
Segmenta el EEG (canal Fpz-Cz) en épocas de 30s etiquetadas por etapa de sueño.
Calcula, por época: entropía de permutación, entropía muestral, dimensión fractal de Higuchi y entropía espectral.
Promedia la complejidad por etapa (W, N1, N2, N3, REM) — típicamente se observa mayor complejidad en vigilia/REM y menor en sueño profundo N3.
Exporta dos CSV.